# Day 6 Demo 3 - Hosting your agent on **Amazon Bedrock AgentCore**

**Read this first.** AgentCore is a **runtime / host**, not a way to *build* agents. You do **not** re-implement the call -> workflow -> agent ladder in AgentCore. You take an agent you already built (Demo 1 hand-rolled, or Demo 2 Strands) and **wrap + host** it, so it runs as a managed, scalable endpoint instead of on your laptop.

This demo shows:
- **A** wrap a single tool-using agent and run it locally,
- **B** wrap a multi-agent system the same way,
- **C** deploy to the cloud (commands; the full click-by-click is in the **Live-AWS Runbook**).

So "the same things with AgentCore" means **the same agents, now hosted** - not a new build pattern.

## Before you run (console + setup, in this order)

1. **Enable model access** - Bedrock console -> **Model catalog** -> **Anthropic Claude Sonnet 4.5** in **us-west-2**.
2. **Install** - `pip install bedrock-agentcore strands-agents bedrock-agentcore-starter-toolkit boto3`.
3. **Credentials** - `aws configure` (or env / role).
4. The wrapper serves on **port 8080** (`POST /invocations`, `GET /ping`). `app.run()` **blocks**, so start the server in a **terminal**, not in a notebook cell.

## A - Wrap a single agent for AgentCore

Three additions turn a Strands agent into a hosted service: `BedrockAgentCoreApp()`, the `@app.entrypoint`, and `app.run()`. The cell below **writes the file** to disk (it does not run a server).

In [ ]:
%%writefile travelmind_agent.py
# travelmind_agent.py - a Strands tool-using agent, wrapped for AgentCore.
from bedrock_agentcore import BedrockAgentCoreApp          # +++ AgentCore wrapper
from strands import Agent, tool
from strands.models import BedrockModel

MODEL = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"     # us. inference profile

@tool
def lookup_booking(pnr: str) -> dict:
    """Look up a booking by its PNR code."""
    return {"pnr": pnr, "status": "CANCELLED", "flight": "AI-302", "date": "2026-06-12"}

@tool
def get_disruption_reason(pnr: str) -> dict:
    """Why a booking was delayed or cancelled."""
    return {"pnr": pnr, "reason": "weather", "detail": "Heavy fog at origin"}

@tool
def get_rebooking_options(pnr: str) -> list:
    """Alternative flights for a disrupted booking."""
    return [{"flight": "AI-318", "dep": "18:40"}, {"flight": "6E-552", "dep": "21:15"}]

model = BedrockModel(model_id=MODEL, region_name="us-west-2")
agent = Agent(model=model,
              tools=[lookup_booking, get_disruption_reason, get_rebooking_options],
              system_prompt="You are TravelMind, a booking-exception assistant. Never invent a PNR.")

app = BedrockAgentCoreApp()                                # +++ create the app

@app.entrypoint                                            # +++ mark the entrypoint
def invoke(payload):
    result = agent(payload.get("prompt", ""))
    return {"result": str(result)}

if __name__ == "__main__":
    app.run()                                              # +++ serve on :8080

**Run it locally (in a terminal, not a cell):**
```
python travelmind_agent.py
```
That starts the server on `:8080` and blocks. In a **second** terminal:
```
curl -X POST http://localhost:8080/invocations -H "Content-Type: application/json" -d '{"prompt": "Status of PNR JX48Q2 and my options?"}'
```
**Expected:** JSON whose `result` mentions the cancellation and the rebooking options.

**Best practices**
- The entrypoint must return a **JSON-serialisable dict** (`{"result": str(result)}`), or invoke returns nothing.
- Test locally before deploying - the same file runs in the cloud.

**Alternatives**
- The hand-rolled agent (Demo 1) wraps identically: replace the Strands body with your loop, keep the three `+++` lines.

## B - Wrap a *multi-agent* system the same way

Hosting does not care how complex the agent is. Here the entrypoint drives the orchestrator (agents-as-tools) from Demo 2.

In [ ]:
%%writefile travelmind_multiagent.py
# travelmind_multiagent.py - an orchestrator (agents-as-tools), wrapped for AgentCore.
from bedrock_agentcore import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel

MODEL = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
model = BedrockModel(model_id=MODEL, region_name="us-west-2")

@tool
def lookup_booking(pnr: str) -> dict:
    """Look up a booking by its PNR code."""
    return {"pnr": pnr, "status": "CANCELLED", "flight": "AI-302", "date": "2026-06-12"}

disruption_agent = Agent(model=model, tools=[lookup_booking],
                         system_prompt="Flight-disruption specialist. Look up the booking, then answer.")
billing_agent    = Agent(model=model,
                         system_prompt="Billing specialist. Be precise about refunds and fees.")

@tool
def handle_disruption(question: str) -> str:
    """Flight delay or cancellation questions."""
    return str(disruption_agent(question))

@tool
def handle_billing(question: str) -> str:
    """Billing, refund, and fee questions."""
    return str(billing_agent(question))

orchestrator = Agent(model=model, tools=[handle_disruption, handle_billing],
                     system_prompt="Route each query to the right specialist tool; reply warmly under 60 words.")

app = BedrockAgentCoreApp()

@app.entrypoint
def invoke(payload):
    return {"result": str(orchestrator(payload.get("prompt", "")))}

if __name__ == "__main__":
    app.run()

**Run locally** exactly like A (`python travelmind_multiagent.py` + the same curl). One endpoint hosts the whole multi-agent system.

**Best practices**
- Keep all agents in one file/process for a single endpoint; split into separate AgentCore agents only when they must scale independently.
- Nested agent loops multiply tokens - watch CloudWatch.

**Alternatives**
- Host each specialist as its own AgentCore agent and let the orchestrator call them via the SDK (more moving parts, independent scaling).

## C - Deploy to the cloud

The starter toolkit deploys the file you wrote. The **full click-by-click + expected output + failure->fix table is in the Live-AWS Runbook**. The short version:
```
agentcore configure -e travelmind_agent.py --disable-memory
agentcore launch                 # provisions the Runtime, prints the Agent ARN
agentcore invoke '{"prompt": "Status of PNR JX48Q2 and my options?"}'
agentcore destroy                # at end of lab
```

**Invoke from an application** (data plane) - replace `<AGENT_ARN>`:
```python
import boto3, json
rt = boto3.client("bedrock-agentcore", region_name="us-west-2")
resp = rt.invoke_agent_runtime(agentRuntimeArn="<AGENT_ARN>",
                               payload=json.dumps({"prompt": "Status of PNR JX48Q2?"}))
print(resp["response"].read().decode())
```

**Watch out**
- Do not mix the **two `agentcore` CLIs** - this is the Python starter toolkit (`configure`/`launch`/`invoke`), not the npm `@aws/agentcore` (`create`/`deploy`).
- Long agent runs need **streaming** or the request times out.
- Idle resources bill - run `agentcore destroy` and delete any Knowledge Bases / OpenSearch collections.

**Coverage note (today).** Day 6 covers the hand-rolled loop (Demo 1), Strands (Demo 2), and this AgentCore intro (Demo 3 + the Runbook). The deeper AgentCore primitives - **Memory, Gateway, Identity, Observability** - are previewed on the slides and go deep on **Day 7**.